# iSTS evaluation on Colab


## Clone the repository

In [ ]:
!git clone https://github.com/nxmbit/ists-finetuning /content/ists-finetuning

## Model settings

Set `MODEL` to a Hugging Face model ID. Leave `SERVER_ARGS` empty to use vLLM defaults.

In [ ]:
from pathlib import Path

REPO_DIR = Path("/content/ists-finetuning")

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_REVISION = None  # Set to a checkpoint commit SHA for a benchmark.
VLLM_VERSION = "0.19.1"
USE_HF_TOKEN = False   # Read HF_TOKEN from Colab Secrets for gated models.

SERVER_PORT = 8000
SERVER_ARGS = []

## Install vLLM

Run this once per Colab runtime or after changing `VLLM_VERSION`.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(
    ["uv", "tool", "install", f"vllm=={VLLM_VERSION}", "--torch-backend=auto"],
    check=True,
)
tool_bin = subprocess.check_output(["uv", "tool", "dir", "--bin"], text=True).strip()
vllm_bin = Path(tool_bin) / "vllm"

## Start vLLM

The first start downloads the checkpoint. Rerun this cell after changing the model
or server settings. Server logs are in `/content/ists-vllm.log`.

In [ ]:
import json
import os
import time
from urllib.error import URLError
from urllib.request import urlopen


def stop_server(process):
    """Stop the vLLM process from this notebook."""
    if process is None or process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()


def wait_for_model(process, api_base, model, log_path):
    """Wait for vLLM to list the requested checkpoint."""
    deadline = time.monotonic() + 900
    while time.monotonic() < deadline:
        if process.poll() is not None:
            tail = log_path.read_text(errors="replace")[-4000:]
            raise RuntimeError(f"vLLM exited. See {log_path}.\n{tail}")
        try:
            with urlopen(f"{api_base}/models", timeout=2) as response:
                models = {entry["id"] for entry in json.load(response)["data"]}
        except (URLError, TimeoutError):
            time.sleep(2)
            continue
        if model not in models:
            raise RuntimeError(f"Expected {model}, but the server lists {sorted(models)}.")
        return
    raise TimeoutError(f"vLLM did not become ready. See {log_path}.")


stop_server(globals().get("server"))
api_base = f"http://127.0.0.1:{SERVER_PORT}/v1"
server_log = Path("/content/ists-vllm.log")
server_env = os.environ.copy()
if USE_HF_TOKEN:
    from google.colab import userdata

    server_env["HF_TOKEN"] = userdata.get("HF_TOKEN")

server_command = [
    str(vllm_bin), "serve", MODEL, "--host", "127.0.0.1", "--port", str(SERVER_PORT),
    *SERVER_ARGS,
]
if MODEL_REVISION:
    server_command.extend(["--revision", MODEL_REVISION])

print(f"Loading {MODEL}. See {server_log} for startup logs.", flush=True)
with server_log.open("w") as log:
    server = subprocess.Popen(
        server_command, cwd=REPO_DIR, env=server_env, stdout=log, stderr=subprocess.STDOUT,
    )
try:
    wait_for_model(server, api_base, MODEL, server_log)
except (Exception, KeyboardInterrupt):
    stop_server(server)
    raise
served_checkpoint = (MODEL, MODEL_REVISION)
print(f"vLLM is ready at {api_base}.")

## Run Evaluation


### Settings

`PROMPTS` accepts a file or a directory. Set `DOMAIN = "all"` for every domain and
`LIMIT = None` for all pairs. `WORKERS` controls concurrent requests.

Approve the Google Drive prompt. To resume, use the same `RUN_DIR` and evaluation settings.

In [ ]:
from google.colab import drive
from pathlib import Path

# Mount Google Drive to access results directory
drive.mount("/content/drive")

PROMPTS = Path("prompts/system/zero-shot.json.txt")
DOMAIN = "images"
LIMIT = 5
WORKERS = 1
RUN_DIR = Path("/content/drive/MyDrive/ists-results/colab-qwen-smoke")

### Evaluation

The first run installs the evaluator dependencies from `uv.lock`.

In [ ]:
if "server" not in globals() or server.poll() is not None or served_checkpoint != (MODEL, MODEL_REVISION):
    raise RuntimeError("Run the vLLM cell for the selected model first.")

prompt_path = REPO_DIR / PROMPTS
if not prompt_path.exists():
    raise FileNotFoundError(prompt_path)
prompt_flag = "--prompt-dir" if prompt_path.is_dir() else "--prompt"

eval_command = [
    "uv", "run", "--locked", "python", "evaluate_ists.py",
    "--model", MODEL,
    prompt_flag, str(PROMPTS),
    "--domain", DOMAIN,
    "--workers", str(WORKERS),
    "--run-dir", str(RUN_DIR)
]
if LIMIT is not None:
    eval_command.extend(["--limit", str(LIMIT)])

eval_env = {**os.environ, "OPENAI_BASE_URL": api_base}
eval_env.pop("OPENAI_API_KEY", None)
subprocess.run(eval_command, cwd=REPO_DIR, env=eval_env, check=True)

## Results

Each CSV contains the domain scores and failure counts. See `review.json` for failed pairs.

In [ ]:
import pandas as pd
from IPython.display import display

results_dir = REPO_DIR / RUN_DIR
pattern = "*/results.csv" if (REPO_DIR / PROMPTS).is_dir() else "results.csv"
for result in sorted(results_dir.glob(pattern)):
    print(result.relative_to(results_dir))
    display(pd.read_csv(result))

## Stop vLLM

Run this when you finish evaluating to release GPU memory.

In [ ]:
stop_server(globals().get("server"))